# =============================================================================
# Example: open-system Heisenberg chain with single-qubit amplitude damping
# =============================================================================


METHODOLOGY NOTE: the dissipator gates built here (via
open_product_formula_generation.jl) use the DIRECT/brute-force route --
dense exponentiation of the local vectorized-Lindbladian generator, no
Kraus operators. This is the baseline to characterize before comparing
against a Kraus-operator-based construction (see kraus_channel_gate in
liouville_space.jl, not yet wired into the product formula).

In [3]:
using ITensors, ITensorMPS
using LinearAlgebra

include("F_diagnostics.jl")   # pulls in everything else via its include chain

F_report (generic function with 1 method)

In [4]:
import Distributions
import Random
Random.seed!(1234)
n = 3                     # number of system qubits
J = rand(Distributions.Uniform(1/4, 3/4), n-1)
gammas = fill(0.05, n)     # uniform amplitude-damping rate on every qubit
t = 3.0                    # total evolution time
ks = [3, 8]                # Trotter step counts to compare (k_i, k_j)
initially_excited = ["0", "2"]   # 0-indexed qubit labels that start in |1>


cutoff = 0
maxdim = 200

lsites = liouville_siteinds(n)
rho0   = vectorized_initial_state_mps(lsites, initially_excited)

3-element MPS:
 ((dim=2|id=482|"Ket,Qubit,n=1"), (dim=2|id=261|"Bra,Qubit,n=1"), (dim=1|id=415|"Link,l=1"))
 ((dim=2|id=16|"Ket,Qubit,n=2"), (dim=2|id=819|"Bra,Qubit,n=2"), (dim=1|id=415|"Link,l=1"), (dim=1|id=117|"Link,l=2"))
 ((dim=2|id=332|"Ket,Qubit,n=3"), (dim=2|id=442|"Bra,Qubit,n=3"), (dim=1|id=117|"Link,l=2"))

In [5]:
# --- Build the full set of F_ij (Eq. 50) for all pairs in ks, via middle-out ---
Fs = build_open_F(n, J, gammas, t, ks, lsites, cutoff, maxdim; order=1)
println("Built ", length(Fs), " F_ij components for ks = ", ks)
for (idx, F) in enumerate(Fs)
    println("  F[$idx]: middle bond dim = ", middle_bond_dim(F))
end

# --- Build F_ex,j (Eq. 51) using a fine reference k0 >> ks ---
k0 = 10
Fex = build_open_F_ex(n, J, gammas, t, ks, k0, lsites, cutoff, maxdim; order=1, order_ref=2)
println("\nBuilt ", length(Fex), " F_ex,j components against reference k0 = ", k0)
for (idx, F) in enumerate(Fex)
    println("  F_ex[$idx]: middle bond dim = ", middle_bond_dim(F))
end

Built 4 F_ij components for ks = [3, 8]
  F[1]: middle bond dim = 16
  F[2]: middle bond dim = 16
  F[3]: middle bond dim = 16
  F[4]: middle bond dim = 16

Built 2 F_ex,j components against reference k0 = 10
  F_ex[1]: middle bond dim = 16
  F_ex[2]: middle bond dim = 16


In [6]:
# --- Track Schmidt-rank growth of F_ii layer by layer (the diagnostic the
#     project notes recommend running first, since the closed-system
#     near-identity argument is not expected to survive once gamma > 0) ---
k_track = 8
result = track_Fii_bond_dimension(
    n, J, gammas, t, k_track, lsites, cutoff, maxdim;
    order=1, track_exact_rank=true, rank_cutoff=1e-10
)

println("\nLayer-by-layer F_ii growth (n=$n, gamma=$(gammas[1]), t=$t, k=$k_track):")
println("layer | stored bond dim | exact operator-Schmidt rank")
for i in eachindex(result.layer)
    println(result.layer[i], "     | ", result.bond_dim[i], "       | ", result.schmidt_rank[i])
end

# Full bond-dimension profile of the final F_ii (all bonds, not just middle)
println("\nFull bond profile of final F_ii: ", full_bond_dimension_profile(result.F_final))



Layer-by-layer F_ii growth (n=3, gamma=0.05, t=3.0, k=8):
layer | stored bond dim | exact operator-Schmidt rank
1     | 16       | 16
2     | 16       | 16
3     | 16       | 16
4     | 16       | 16
5     | 16       | 16
6     | 16       | 16
7     | 16       | 16
8     | 16       | 16

Full bond profile of final F_ii: [16, 16]


In [7]:
# =============================================================================
# F(t) report: <<rho(0)|F_ij|rho(0)>> matrix and distance-to-identity,
# mirroring Eqs. 14-16 of main.pdf
# =============================================================================
#
# OPEN system (dissipation on): expect the diagonal entries to be visibly
# below 1 (and ||F_ii - Id||_F visibly above 0), unlike the closed-system
# case where F_ii is exactly the identity by construction.
 
report_open = F_report(
    n, J, gammas, t, ks, initially_excited, lsites, cutoff, maxdim;
    order=1, dissipation=true
)
 
println("\n--- Open system (dissipation ON, gamma = $(gammas[1])) ---")
println("<<rho(0)|F_ij|rho(0)>> matrix:")
display(report_open.M)
println("\n||F_ij - Id||_F matrix:")
display(report_open.distances)
 
# CLOSED system check (dissipation off): set dissipation=false to recover
# the ideal unitary Heisenberg evolution with NO code changes elsewhere --
# expect the diagonal of M to be numerically 1 and the diagonal of
# `distances` to be numerically 0, exactly as in the closed-system paper's
# Eq. 14 (e.g. F11 = F22 = 0.999999 there).
 
report_closed = F_report(
    n, J, gammas, t, ks, initially_excited, lsites, cutoff, maxdim;
    order=1, dissipation=false
)
 
println("\n--- Closed-system check (dissipation OFF) ---")
println("<<rho(0)|F_ij|rho(0)>> matrix:")
display(report_closed.M)
println("\n||F_ij - Id||_F matrix:")
display(report_closed.distances)

2×2 Matrix{ComplexF64}:
 0.570296+3.72237e-16im  0.535613+1.572e-16im
 0.535613+9.8591e-16im   0.569789-1.18433e-15im


--- Open system (dissipation ON, gamma = 0.05) ---
<<rho(0)|F_ij|rho(0)>> matrix:

||F_ij - Id||_F matrix:

--- Closed-system check (dissipation OFF) ---
<<rho(0)|F_ij|rho(0)>> matrix:

||F_ij - Id||_F matrix:


2×2 Matrix{Float64}:
 3.02031  3.66146
 3.66146  3.0181

2×2 Matrix{ComplexF64}:
      1.0-6.84317e-16im  0.938085-2.51936e-16im
 0.938085+3.2488e-15im        1.0+1.73151e-15im

2×2 Matrix{Float64}:
 5.44764e-14  2.57475
 2.57475      5.0781e-14

# =============================================================================
# Closed-system sanity check (dissipation OFF)
# Expect: purity = 1.0, M diagonal = 1.0, E_mpf < E_trot[j] for all j
# =============================================================================

In [9]:
include("open_optimization_problem.jl")   # pulls in the full include chain

result_closed = test_dynamic_mpf_open(
    n, J, gammas, t, ks, 10, 100, lsites, rho0;
    cutoff_opt=0,  maxdim_opt=50,
    cutoff_eval=0, maxdim_eval=200,
    order=1, order_ref_opt=1, order_ref_eval=2,
    dissipation=false
)
 
println("\n=== Closed-system check (dissipation OFF) ===")
println("\nGram matrix M_opt:")
display(result_closed.M_opt)
println("\nOverlap vector L_opt:")
display(result_closed.L_opt)
println("\nMPF coefficients c: ", result_closed.coeffs)
println("Sum of coefficients: ", sum(result_closed.coeffs))
println("\nPurity Tr(rho^2(t)) = ", result_closed.purity, "  [should be ~1.0]")
println("\nDMPF error E_mpf  = ", result_closed.E_mpf)
println("Single-Trotter errors E_kj = ", result_closed.E_trot)

2×2 Matrix{Float64}:
 1.0       0.938085
 0.938085  1.0

2-element Vector{Float64}:
 0.924095231528477
 0.9992282131861707


=== Closed-system check (dissipation OFF) ===

Gram matrix M_opt:

Overlap vector L_opt:

MPF coefficients c: [-0.10674775783879455, 1.1067477578387945]
Sum of coefficients: 1.0

Purity Tr(rho^2(t)) = 0.9999999999992186  [should be ~1.0]

DMPF error E_mpf  = 0.024955149119224584
Single-Trotter errors E_kj = [0.28395183068534635, 0.03671736176081386]


# =============================================================================
# Full open-system MPF workflow (dissipation ON)
# =============================================================================

In [10]:
result = test_dynamic_mpf_open(
    n, J, gammas, t, ks, 10, 100, lsites, rho0;
    cutoff_opt=0,  maxdim_opt=50,
    cutoff_eval=0, maxdim_eval=200,
    order=1, order_ref_opt=1, order_ref_eval=2,
    dissipation=true
)
 
println("=== Open system (dissipation ON, gamma = $(gammas[1])) ===")
println("\nGram matrix M_opt (Eq. 46):")
display(result.M_opt)
println("\nOverlap vector L_opt (Eq. 47):")
display(result.L_opt)
println("\nMPF coefficients c (Eq. 40): ", result.coeffs)
println("Lagrange multiplier lambda:  ", result.lambda)
println("Sum of coefficients (should be 1): ", sum(result.coeffs))
println("\nReference Gram matrix M_ref:")
display(result.M_ref)
println("\nReference overlap vector L_ref:")
display(result.L_ref)
println("\nPurity Tr(rho^2(t)) = ", result.purity, "  [closed system would be 1.0]")
println("\nDMPF error E_mpf  = ", result.E_mpf)
println("Single-Trotter errors E_kj = ", result.E_trot)

2×2 Matrix{Float64}:
 0.570296  0.535613
 0.535613  0.569789

2-element Vector{Float64}:
 0.5278454260804764
 0.5693491457829063

2×2 Matrix{Float64}:
 0.570296  0.535613
 0.535613  0.569789

2-element Vector{Float64}:
 0.4912807472560178
 0.5595871194149613

=== Open system (dissipation ON, gamma = 0.05) ===

Gram matrix M_opt (Eq. 46):

Overlap vector L_opt (Eq. 47):

MPF coefficients c (Eq. 40): [-0.10640815303059248, 1.1064081530305925]
Lagrange multiplier lambda:  0.004076470571201674
Sum of coefficients (should be 1): 1.0

Reference Gram matrix M_ref:

Reference overlap vector L_ref:

Purity Tr(rho^2(t)) = 0.5697224274776859  [closed system would be 1.0]

DMPF error E_mpf  = 0.013853429355999314
Single-Trotter errors E_kj = [0.15745711472658752, 0.020337152802255787]


In [ ]:
include("bond_dimension_scaling_study.jl")   # pulls in the full include chain

# ---- experiment 1: layer growth at fixed n ----
n1 = 5
gammas_list = [0.01, 0.05, 0.1, 0.2]
t = 3.0
k = 8
cutoff = 1e-10
maxdim1 = 4000          # generous relative to chi_max(5)=16^2=256

results, chi_max1 = layer_growth_study(n1, gammas_list, t, k; cutoff=cutoff, maxdim=maxdim1)
plt1 = plot_layer_growth(n1, gammas_list, results, chi_max1; maxdim=maxdim1)
# savefig(plt1, "layer_growth_n$(n1).png")

open("layer_growth_n$(n1).csv", "w") do io
    println(io, "layer,", join(["gamma_$g" for g in sort(gammas_list)], ","))
    for layer in 1:k
        row = [string(layer); [string(results[g][layer]) for g in sort(gammas_list)]]
        println(io, join(row, ","))
    end
end

# ---- experiment 2: scaling with n ----
ns = [3, 4, 5, 6]        # extend cautiously -- cost grows like 16^(n/2)
maxdim2 = 4000            # chi_max(6) = 4096; already brushing the cap at n=6

ns_out, final_bd, capped, chi_max2 = n_scaling_study(ns, gammas_list, t, k;
                                                        cutoff=cutoff, maxdim=maxdim2)
plt2 = plot_n_scaling(ns_out, gammas_list, final_bd, capped, chi_max2; maxdim=maxdim2)
savefig(plt2, "n_scaling.png")

open("n_scaling.csv", "w") do io
    println(io, "n,", join(["gamma_$g" for g in sort(gammas_list)], ","), ",theoretical_max")
    for (idx, n) in enumerate(ns_out)
        row = [string(n); [string(final_bd[g][idx]) for g in sort(gammas_list)]; string(chi_max2[idx])]
        println(io, join(row, ","))
    end
end

println("\nSaved: layer_growth_n$(n1).png, layer_growth_n$(n1).csv, n_scaling.png, n_scaling.csv")

n=5, gamma=0.01: bond_dim by layer = [13, 76, 146, 214, 238, 248, 252, 254]
n=5, gamma=0.05: bond_dim by layer = [13, 107, 210, 241, 250, 254, 255, 256]
n=5, gamma=0.1: bond_dim by layer = [13, 138, 231, 246, 252, 254, 256, 256]
n=5, gamma=0.2: bond_dim by layer = [13, 177, 239, 250, 254, 256, 256, 256]
n=3, gamma=0.01: final bond dim (layer 8) = 16   (theoretical max = 16)
n=3, gamma=0.05: final bond dim (layer 8) = 16   (theoretical max = 16)
n=3, gamma=0.1: final bond dim (layer 8) = 16   (theoretical max = 16)
n=3, gamma=0.2: final bond dim (layer 8) = 16   (theoretical max = 16)
n=4, gamma=0.01: final bond dim (layer 8) = 240   (theoretical max = 256)
n=4, gamma=0.05: final bond dim (layer 8) = 254   (theoretical max = 256)
n=4, gamma=0.1: final bond dim (layer 8) = 256   (theoretical max = 256)
n=4, gamma=0.2: final bond dim (layer 8) = 256   (theoretical max = 256)
n=5, gamma=0.01: final bond dim (layer 8) = 250